# BERT for Aspect-Based Sentiment Analysis (ABSA)

This notebook contains the complete code for BERT-based Aspect-Based Sentiment Analysis. It combines all the Python modules into a single notebook.

## Contents:
1. **Imports & Configuration**
2. **Data Loading** (loader.py)
3. **Tokenizer** (Tokenizer.py)
4. **Models** (model.py)
5. **Training & Evaluation** (train.py)
6. **Main Pipeline** (main.py)

## 1. Imports & Configuration

In [1]:
from __future__ import print_function
import random
import copy
import os
from ast import literal_eval
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as utils
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import BertModel, BertForSequenceClassification, BertTokenizer

# Set random seeds for reproducibility
SEED = 1345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Model hyperparameters
fchidden = 256
hiddendim_lstm = 128
embeddim = 768
numlayers = 12

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

d:\System\Conda\envs\absa\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 2. Data Loading Functions

Functions to load SemEval and Twitter datasets for ABSA.

In [ ]:
'''
def load_twitter(dataset, data_path=None):
    """Loads Twitter dataset

    Arguments:
        dataset (str): Name of the dataset
        data_path (str): Path to the dataset

    """
    if(data_path is None):
        data_path = '../Data/'
    train_file = data_path + dataset + '/train.raw'
    test_file = data_path + dataset + '/test.raw'

    train_sentence = []
    train_aspect = []
    train_sentiment = []

    with open(train_file, 'r', encoding='latin1') as f:
        lines = f.readlines()

    for i in range(0, len(lines), 3):
        if(lines[i+2][:-1] == '-1'):
            lines[i+2] = '2'
        curind = lines[i].find('$T$')
        asp = lines[i+1][1]
        sen = lines[i][0: curind] + " " + asp + " " + lines[i][curind + 3: -1]
        train_sentence.append(sen)
        train_aspect.append(asp)
        train_sentiment.append(literal_eval(lines[i + 2]))

    test_sentence = []
    test_aspect = []
    test_sentiment = []

    with open(test_file, 'r', encoding='latin1') as f:
        lines = f.readlines()

    for i in range(0, len(lines), 3):
        if(lines[i+2][:-1] == '-1'):
            lines[i+2] = '2'
        curind = lines[i].find('$T$')
        asp = lines[i+1][1]
        sen = lines[i][0: curind] + " " + asp + " " + lines[i][curind + 3: -1]
        test_sentence.append(sen)
        test_aspect.append(asp)
        test_sentiment.append(literal_eval(lines[i+2]))

    train_sen_len = [len(sentence.split()) for sentence in train_sentence]
    train_asp_len = [len(aspect.split()) for aspect in train_aspect]

    print("----------------------------------------")
    print("Maximum Training data Sentence Length: {}".format(
                                                       max(train_sen_len)))
    print("Maximum Training data Aspect Length: {}".format(max(train_asp_len)))
    print("----------------------------------------")

    return (train_sentence, train_aspect, train_sentiment, test_sentence,
            test_aspect, test_sentiment)

            '''

In [3]:
def load_semeval(dataset, num_classes, data_path=None):
    """ Loads SemEval 14 datasets

    Arguments:
        dataset (str): Name of the datsets
        num_classes (int): Number of classes to consider
        data_path (str): Path to the dataset

    """
    label = {'negative': 0,
             'positive': 1,
             'neutral': 2,
             'conflict': 3}

    if(data_path is None):
        data_path = '../Data/'
    train_file = data_path + 'atsa-' + dataset + '/atsa_train.json'
    test_file = data_path + 'atsa-' + dataset + '/atsa_test.json'

    temp = open(train_file, 'r', encoding='latin1').read()
    train = literal_eval(temp)
    train_sentence = []
    train_aspect = []
    train_sentiment = []
    for xml in train:
        if(xml['sentiment'] == 'conflict' and num_classes == 3):
            continue
        train_sentence.append(xml['sentence'])
        train_aspect.append(xml['aspect'])
        train_sentiment.append(label[xml['sentiment']])

    temp = open(test_file, 'r', encoding='latin1').read()
    test = literal_eval(temp)
    test_sentence = []
    test_aspect = []
    test_sentiment = []
    for xml in test:
        if(xml['sentiment'] == 'conflict' and num_classes == 3):
            continue
        test_sentence.append(xml['sentence'])
        test_aspect.append(xml['aspect'])
        test_sentiment.append(label[xml['sentiment']])

    train_sen_len = [len(sentence.split()) for sentence in train_sentence]
    train_asp_len = [len(aspect.split()) for aspect in train_aspect]

    print("----------------------------------------")
    print("Maximum Training data Sentence Length: {}".format(
                                                       max(train_sen_len)))
    print("Maximum Training data Aspect Length: {}".format(max(train_asp_len)))
    print("----------------------------------------")

    return (train_sentence, train_aspect, train_sentiment, test_sentence,
            test_aspect, test_sentiment)

In [4]:
def get_loader(input_ids, attention_masks, token_type_ids, labels, batchsize):
    """ Converts input values into a dataloader

    Arguments:
    input_ids (Tensors): Sentences converted to input ids
    attention_masks (Tensors): Attention masks of the words in a sentence
    token_type_ids (Tensors): Token ids for pair of sentences
    labels (Tensors): Labels of the Dataset
    batchsize (int): Batch size to train

    """
    array = utils.TensorDataset(input_ids, attention_masks, token_type_ids,
                                labels)
    loader = utils.DataLoader(array, batch_size=batchsize)
    return loader

## 3. Tokenizer Functions

Functions to load pretrained BERT tokenizer and tokenize sentences.

In [5]:
def get_pretrained_tokenizer():
    """
    Loads pretrained BERT Tokenizer and returns it,

    """
    print("Downloading bert tokenizer to cache")
    print("---------------------------------------")
    bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased',
                                                   do_lower_case=True
                                                   )
    return bert_tokenizer


def tokenize_sentences(bert_tokenizer, sentences, aspects, maxlen):
    """ converts sentences into ids according to bert tokenizer

    Arguments:
    bert_tokenizer (Tokenizer): Pretrained BERT Tokenizer
    sentences (list): List of sentences
    aspects (list): List of Aspects
    maxlen (int): Maximum Length of a sentence

    """
    input_ids = []
    attention_masks = []
    token_type_ids = []

    for sentence, aspect in zip(sentences, aspects):

        encoded = bert_tokenizer.encode_plus(text=sentence,
                                             text_pair=aspect,
                                             add_special_tokens=True,
                                             max_length=maxlen,
                                             padding='max_length',
                                             truncation=True,
                                             return_attention_mask=True,
                                             return_token_type_ids=True,
                                             return_tensors='pt')

        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])
        token_type_ids.append(encoded['token_type_ids'])

    input_ids = torch.cat(input_ids, dim=0, out=None)
    attention_masks = torch.cat(attention_masks, dim=0, out=None)
    token_type_ids = torch.cat(token_type_ids, dim=0, out=None)

    return input_ids, attention_masks, token_type_ids

## 4. Model Definitions

Three BERT-based models for ABSA:
- **Bert_Base**: Using BertForSequenceClassification
- **Bert_LSTM**: BERT + LSTM
- **Bert_Attention**: BERT + Attention mechanism

In [6]:
class Bert_Base(nn.Module):
    def __init__(self, numclasses):
        super(Bert_Base, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.dropout = nn.Dropout(0.1)

        self.bert = BertForSequenceClassification.from_pretrained('bert-base-uncased',
                                                                   output_hidden_states=False,
                                                                   output_attentions=False,
                                                                   num_labels=self.numclasses)
        print("BERT Model Loaded")

    def forward(self, inp_ids, att_mask, token_ids, labels):
        outputs = self.bert(input_ids=inp_ids, attention_mask=att_mask,
                            token_type_ids=token_ids, labels=labels)
        return outputs.loss, outputs.logits

In [7]:
class Bert_LSTM(nn.Module):
    def __init__(self, numclasses):
        super(Bert_LSTM, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.numlayers = numlayers
        self.hiddendim_lstm = hiddendim_lstm
        self.dropout = nn.Dropout(0.1)

        self.bert = BertModel.from_pretrained('bert-base-uncased',
                                              output_hidden_states=True,
                                              output_attentions=False)
        print("BERT Model Loaded")
        self.lstm = nn.LSTM(self.embeddim, self.hiddendim_lstm, batch_first=True)
        self.fc = nn.Linear(self.hiddendim_lstm, self.numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(input_ids=inp_ids,
                            attention_mask=att_mask,
                            token_type_ids=token_ids)
        hidden_states = outputs.hidden_states

        hidden_states = torch.stack([hidden_states[layer_i][:, 0].squeeze()
                                     for layer_i in range(0, self.numlayers)], dim=-1)
        hidden_states = hidden_states.view(-1, self.numlayers, self.embeddim)
        out, _ = self.lstm(hidden_states, None)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

In [8]:
class Bert_Attention(nn.Module):
    def __init__(self, numclasses, device=None):
        super(Bert_Attention, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.numlayers = numlayers
        self.fchidden = fchidden
        self.dropout = nn.Dropout(0.1)

        self.bert = BertModel.from_pretrained('bert-base-uncased',
                                              output_hidden_states=True,
                                              output_attentions=False)
        print("BERT Model Loaded")

        # Initialize attention parameters - don't use .to(device) here
        # Let PyTorch handle device placement when model.to(device) is called
        q_t = np.random.normal(loc=0.0, scale=0.1, size=(1, self.embeddim))
        self.q = nn.Parameter(torch.from_numpy(q_t).float())
        
        w_ht = np.random.normal(loc=0.0, scale=0.1, size=(self.embeddim, self.fchidden))
        self.w_h = nn.Parameter(torch.from_numpy(w_ht).float())

        self.fc = nn.Linear(self.fchidden, self.numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(input_ids=inp_ids,
                            attention_mask=att_mask,
                            token_type_ids=token_ids)
        hidden_states = outputs.hidden_states

        hidden_states = torch.stack([hidden_states[layer_i][:, 0].squeeze()
                                     for layer_i in range(0, self.numlayers)], dim=-1)
        hidden_states = hidden_states.view(-1, self.numlayers, self.embeddim)
        out = self.attention(hidden_states)
        out = self.dropout(out)
        out = self.fc(out)
        return out

    def attention(self, h):
        v = torch.matmul(self.q, h.transpose(-2, -1)).squeeze(1)
        v = F.softmax(v, -1)
        v_temp = torch.matmul(v.unsqueeze(1), h).transpose(-2, -1)
        v = torch.matmul(self.w_h.transpose(1, 0), v_temp).squeeze(2)
        return v

## 5. Training & Evaluation Functions

Functions for training the model and evaluating performance.

In [9]:
def evaluate(loader, net, device, model_name):
    """ Evaluates a model and returns loss, accuracy

    Arguments:
    loader (DataLoader): dataloader to evaluate
    net (nn.Module): Model to evaluate
    device (torch.device): Device type
    model_name (str): Model name

    """
    net.eval()

    with torch.no_grad():
        loss = 0.0
        total = 0.0
        acc = 0.0
        y_pred = []
        y_true = []
        for input_id, attention_masks, token_ids, labels in loader:
            input_id = input_id.to(device)
            attention_masks = attention_masks.to(device)
            token_ids = token_ids.to(device)
            labels = labels.long().to(device)

            if(model_name == 'base'):
                curloss, output = net(input_id, attention_masks, token_ids, labels)
            else:
                output = net(input_id, attention_masks, token_ids)
                curloss = F.cross_entropy(output, labels, reduction='sum')
            loss += curloss.item()
            preds = torch.argmax(output, 1)
            y_pred.extend(preds.tolist())
            y_true.extend(labels.tolist())
            acc += torch.sum(preds == labels).item()
            total += input_id.size(0)

        F1 = round((f1_score(y_true, y_pred, average='macro')), 2) * 100
        return round((loss / total), 3), round(((acc / total) * 100), 2), F1

In [10]:
def train_model(train_loader, dev_loader, test_loader, model_name,
                numclasses, numepochs, runs, device):
    """ Trains the neural network

    Arguments:
    train_loader (DataLoader): Training Data Loader
    dev_loader (DataLoader): Validation Data Loader
    test_loader (DataLoader): Test Data Loader
    model_name (str): Name of the model to train
    numclasses (int): Number of classes in the data
    numepochs (int): Number of epochs to train
    runs (int): Number of runs to report averaged results
    device (torch.device): Device type

    Returns:
    best_model: The best trained model from the last run
    """
    avg_testacc = 0.0
    avg_testf1 = 0.0
    best_overall_state_dict = None
    best_overall_acc = 0.0
    
    for run in range(1, runs+1):
        print("Training for run {} ".format(run))
        print("--------------------------------------------")
        if(model_name == 'lstm'):
            model = Bert_LSTM(numclasses).to(device)
        elif(model_name == 'attention'):
            model = Bert_Attention(numclasses, device).to(device)
        else:
            model = Bert_Base(numclasses).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

        model.train()
        valbest = 0.0
        best_model_wts = copy.deepcopy(model.state_dict())

        for epoch in range(1, numepochs+1):
            model.train()
            for input_id, attention_masks, token_ids, labels in train_loader:
                input_id = input_id.to(device)
                attention_masks = attention_masks.to(device)
                token_ids = token_ids.to(device)
                labels = labels.long().to(device)

                model.zero_grad()

                if(model_name == 'base'):
                    loss, _ = model(input_id, attention_masks, token_ids, labels)
                else:
                    output = model(input_id, attention_masks, token_ids)
                    loss = F.cross_entropy(output, labels)

                loss.backward()
                optimizer.step()

            valloss, valacc, _ = evaluate(dev_loader, model, device, model_name)
            if(valacc > valbest):
                valbest = valacc
                best_model_wts = copy.deepcopy(model.state_dict())

            print("Epoch {} Val Loss {} Val Acc {} ".format(epoch,
                                                            valloss,
                                                            valacc))

        model.load_state_dict(best_model_wts)

        curtestloss, curtestacc, curtestf1 = evaluate(test_loader,
                                                      model, device,
                                                      model_name)

        print("Run {} Test Accuracy {} F1 Score {}".format(run,
                                                           curtestacc,
                                                           curtestf1))
        print("---------------------------------------------------")
        avg_testacc += curtestacc
        avg_testf1 += curtestf1
        
        # Keep track of the best model state dict across all runs
        if curtestacc > best_overall_acc:
            best_overall_acc = curtestacc
            best_overall_state_dict = copy.deepcopy(model.state_dict())

    print("Average Test Accuracy: {} F1: {} ".format(avg_testacc/runs,
                                                     avg_testf1/runs))
    
    # Recreate the best model and load the best state dict
    if(model_name == 'lstm'):
        best_model = Bert_LSTM(numclasses).to(device)
    elif(model_name == 'attention'):
        best_model = Bert_Attention(numclasses, device).to(device)
    else:
        best_model = Bert_Base(numclasses).to(device)
    
    best_model.load_state_dict(best_overall_state_dict)
    best_model.eval()
    
    return best_model

## 6. Main Pipeline

Configure the parameters and run the training pipeline.

In [11]:
# Configuration parameters (modify these as needed)
dataset = 'laptop'          # Options: 'laptop', 'restaurant', 'twitter'
maxlen = 80                 # Maximum sentence length for BERT
numclasses = 3              # Number of classes (3 or 4 to include 'conflict')
model_name = 'lstm'         # Options: 'lstm', 'attention', 'base'
data_path = '../Data/'      # Path to dataset folder
batch_size = 8              # Batch size
numepochs = 10               # Number of training epochs (reduced for faster testing)
runs = 10                    # Number of runs (reduced for faster testing)

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.59 GB


In [ ]:
# Load dataset
if dataset in ['laptop', 'restaurant']:
    print('semeval')
    (train_sentence, train_aspect, train_sentiment,
     test_sentence, test_aspect, test_sentiment) = load_semeval(dataset,
                                                                numclasses,
'''                                                                data_path)
elif dataset == 'twitter':
    print('twitter')
    (train_sentence, train_aspect, train_sentiment,
     test_sentence, test_aspect, test_sentiment) = load_twitter(dataset,
                                                                data_path)'''
else:
    raise ValueError('Invalid dataset selected. Choose from: laptop, restaurant, twitter')

semeval
----------------------------------------
Maximum Training data Sentence Length: 78
Maximum Training data Aspect Length: 6
----------------------------------------


In [13]:
# Split training data into train and validation sets
(train_sentence, dev_sentence, train_aspect, dev_aspect,
 train_sentiment, dev_sentiment) = train_test_split(train_sentence,
                                                    train_aspect,
                                                    train_sentiment,
                                                    test_size=0.1,
                                                    random_state=42)

print("Training Data size: {}".format(len(train_sentence)))
print("Validation Data size: {}".format(len(dev_sentence)))
print("Test Data size: {}".format(len(test_sentence)))
print("------------------------------------------------")

Training Data size: 2081
Validation Data size: 232
Test Data size: 638
------------------------------------------------


In [14]:
# Load pretrained BERT tokenizer
bert_tokenizer = get_pretrained_tokenizer()

---------------------------------------


In [15]:
# Tokenize training data
print("Tokenizing training data")
(train_input_ids, train_attention_masks,
 train_token_type_ids) = tokenize_sentences(bert_tokenizer,
                                            train_sentence,
                                            train_aspect,
                                            maxlen)
train_labels = torch.from_numpy(np.asarray(train_sentiment, 'int32'))

# Tokenize validation data
print("Tokenizing validation data")
(dev_input_ids, dev_attention_masks,
 dev_token_type_ids) = tokenize_sentences(bert_tokenizer,
                                          dev_sentence,
                                          dev_aspect,
                                          maxlen)
dev_labels = torch.from_numpy(np.asarray(dev_sentiment, 'int32'))

# Tokenize test data
print("Tokenizing test data")
(test_input_ids, test_attention_masks,
 test_token_type_ids) = tokenize_sentences(bert_tokenizer,
                                           test_sentence,
                                           test_aspect,
                                           maxlen)
test_labels = torch.from_numpy(np.asarray(test_sentiment, 'int32'))
print("-------------------------------------------------")

Tokenizing training data


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Tokenizing validation data
Tokenizing test data


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

-------------------------------------------------


In [16]:
# Create data loaders
train_loader = get_loader(train_input_ids, train_attention_masks,
                          train_token_type_ids, train_labels,
                          batch_size)
dev_loader = get_loader(dev_input_ids, dev_attention_masks,
                        dev_token_type_ids, dev_labels,
                        batch_size)
test_loader = get_loader(test_input_ids, test_attention_masks,
                         test_token_type_ids, test_labels,
                         batch_size)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(dev_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 261
Validation batches: 29
Test batches: 80


In [17]:
# Train all 3 models and store them (skip if already saved)
model_names = ['lstm', 'attention', 'base']
trained_models = {}  # Dictionary to store all trained models
model_results = {}   # Dictionary to store results
saved_model_paths = {}  # Dictionary to store saved paths

for model_name in model_names:
    model_save_path = f'bert_absa_{model_name}_model.pth'
    
    # Check if model already exists
    if os.path.exists(model_save_path):
        print("\n" + "=" * 70)
        print(f"SKIPPING {model_name.upper()} MODEL - Already saved at: {model_save_path}")
        print("=" * 70)
        
        # Load existing model for evaluation
        checkpoint = torch.load(model_save_path, map_location=DEVICE)
        if model_name == 'lstm':
            trained_model = Bert_LSTM(numclasses).to(DEVICE)
        elif model_name == 'attention':
            trained_model = Bert_Attention(numclasses, DEVICE).to(DEVICE)
        else:
            trained_model = Bert_Base(numclasses).to(DEVICE)
        trained_model.load_state_dict(checkpoint['model_state_dict'])
        trained_model.eval()
    else:
        print("\n" + "=" * 70)
        print(f"TRAINING {model_name.upper()} MODEL")
        print("=" * 70)
        
        # Train the model
        trained_model = train_model(train_loader, dev_loader, test_loader, model_name,
                                    numclasses, numepochs, runs, DEVICE)
        
        # Save immediately after training
        torch.save({
            'model_state_dict': trained_model.state_dict(),
            'model_type': model_name,
            'numclasses': numclasses,
            'maxlen': maxlen,
        }, model_save_path)
        print(f"\n✓ Saved {model_name.upper()} model to: {model_save_path}")
    
    # Store the trained model and path
    trained_models[model_name] = trained_model
    saved_model_paths[model_name] = model_save_path
    
    # Evaluate and store results
    train_loss, train_acc, train_f1 = evaluate(train_loader, trained_model, DEVICE, model_name)
    val_loss, val_acc, val_f1 = evaluate(dev_loader, trained_model, DEVICE, model_name)
    test_loss, test_acc, test_f1 = evaluate(test_loader, trained_model, DEVICE, model_name)
    
    model_results[model_name] = {
        'train_acc': train_acc, 'train_f1': train_f1,
        'val_acc': val_acc, 'val_f1': val_f1,
        'test_acc': test_acc, 'test_f1': test_f1
    }

# Print comparison of all models
print("\n" + "=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)
print(f"\n{'Model':<12} {'Train Acc':<12} {'Val Acc':<12} {'Test Acc':<12} {'Test F1':<12}")
print("-" * 60)
for name, results in model_results.items():
    print(f"{name.upper():<12} {results['train_acc']:<12} {results['val_acc']:<12} {results['test_acc']:<12} {results['test_f1']:<12}")

# Find best model
best_model_name = max(model_results, key=lambda x: model_results[x]['test_acc'])
print(f"\nBest performing model: {best_model_name.upper()} (Test Acc: {model_results[best_model_name]['test_acc']}%)")

print("\n" + "=" * 70)
print("SAVED MODEL FILES")
print("=" * 70)
for name, path in saved_model_paths.items():
    print(f"  ✓ {name.upper()}: {path}")


SKIPPING LSTM MODEL - Already saved at: bert_absa_lstm_model.pth
BERT Model Loaded

SKIPPING ATTENTION MODEL - Already saved at: bert_absa_attention_model.pth
BERT Model Loaded

SKIPPING BASE MODEL - Already saved at: bert_absa_base_model.pth


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT Model Loaded

MODEL COMPARISON SUMMARY

Model        Train Acc    Val Acc      Test Acc     Test F1     
------------------------------------------------------------
LSTM         99.23        79.31        77.43        73.0        
ATTENTION    99.28        80.6         79.15        75.0        
BASE         98.94        78.45        77.74        73.0        

Best performing model: ATTENTION (Test Acc: 79.15%)

SAVED MODEL FILES
  ✓ LSTM: bert_absa_lstm_model.pth
  ✓ ATTENTION: bert_absa_attention_model.pth
  ✓ BASE: bert_absa_base_model.pth


## 7. Save Models

Save all 3 trained models to separate files.

In [18]:
# Save all 3 trained models to separate files
saved_model_paths = {}

for model_name, model in trained_models.items():
    model_save_path = f'bert_absa_{model_name}_model.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_type': model_name,
        'numclasses': numclasses,
        'maxlen': maxlen,
    }, model_save_path)
    saved_model_paths[model_name] = model_save_path
    print(f"✓ Saved {model_name.upper()} model to: {model_save_path}")

print("\nAll 3 models saved successfully!")
print("\nSaved files:")
for name, path in saved_model_paths.items():
    print(f"  - {path}")

✓ Saved LSTM model to: bert_absa_lstm_model.pth
✓ Saved ATTENTION model to: bert_absa_attention_model.pth
✓ Saved BASE model to: bert_absa_base_model.pth

All 3 models saved successfully!

Saved files:
  - bert_absa_lstm_model.pth
  - bert_absa_attention_model.pth
  - bert_absa_base_model.pth


## 8. Inference

Load models from saved files and perform inference.

In [19]:
# Function to load a saved model
def load_model(model_path, device):
    """Load a saved model for inference."""
    checkpoint = torch.load(model_path, map_location=device)
    
    model_type = checkpoint['model_type']
    num_classes = checkpoint['numclasses']
    max_len = checkpoint['maxlen']
    
    if model_type == 'lstm':
        model = Bert_LSTM(num_classes).to(device)
    elif model_type == 'attention':
        model = Bert_Attention(num_classes, device).to(device)
    else:
        model = Bert_Base(num_classes).to(device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    return model, max_len, model_type


# Prediction function
def predict_sentiment(model, bert_tokenizer, sentence, aspect, maxlen, device, model_type='lstm'):
    """Predict sentiment for a sentence-aspect pair."""
    model.eval()
    
    encoded = bert_tokenizer.encode_plus(
        text=sentence,
        text_pair=aspect,
        add_special_tokens=True,
        max_length=maxlen,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors='pt'
    )
    
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    token_type_ids = encoded['token_type_ids'].to(device)
    
    with torch.no_grad():
        if model_type == 'base':
            dummy_label = torch.tensor([0]).to(device)
            _, output = model(input_ids, attention_mask, token_type_ids, dummy_label)
        else:
            output = model(input_ids, attention_mask, token_type_ids)
        
        probabilities = F.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    
    return predicted.item(), confidence.item()


# Label mapping
label_names = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}

### 8.1 LSTM Model Inference

In [20]:
# Load LSTM model from saved file
lstm_model_path = 'bert_absa_lstm_model.pth'
lstm_model, lstm_maxlen, lstm_type = load_model(lstm_model_path, DEVICE)
print(f"✓ Loaded LSTM model from: {lstm_model_path}")

# Test examples
test_examples = [
    ("The battery life of this laptop is amazing and lasts all day.", "battery life"),
    ("The screen resolution is terrible and hurts my eyes.", "screen resolution"),
    ("The keyboard is okay, nothing special.", "keyboard"),
    ("I love the fast performance of this machine.", "performance"),
    ("The customer service was unhelpful and rude.", "customer service"),
]

print("\n" + "=" * 70)
print("LSTM MODEL PREDICTIONS")
print("=" * 70)

for sentence, aspect in test_examples:
    pred_label, confidence = predict_sentiment(lstm_model, bert_tokenizer,
                                                sentence, aspect, lstm_maxlen, DEVICE,
                                                model_type=lstm_type)
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print(f"Prediction: {label_names[pred_label]} (confidence: {confidence:.2%})")

BERT Model Loaded
✓ Loaded LSTM model from: bert_absa_lstm_model.pth

LSTM MODEL PREDICTIONS

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
Prediction: Positive (confidence: 99.50%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
Prediction: Negative (confidence: 98.38%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
Prediction: Positive (confidence: 99.34%)

Sentence: "I love the fast performance of this machine."
Aspect: "performance"
Prediction: Positive (confidence: 99.50%)

Sentence: "The customer service was unhelpful and rude."
Aspect: "customer service"
Prediction: Negative (confidence: 98.02%)


### 8.2 Attention Model Inference

In [21]:
# Load Attention model from saved file
attention_model_path = 'bert_absa_attention_model.pth'
attention_model, attention_maxlen, attention_type = load_model(attention_model_path, DEVICE)
print(f"✓ Loaded ATTENTION model from: {attention_model_path}")

print("\n" + "=" * 70)
print("ATTENTION MODEL PREDICTIONS")
print("=" * 70)

for sentence, aspect in test_examples:
    pred_label, confidence = predict_sentiment(attention_model, bert_tokenizer,
                                                sentence, aspect, attention_maxlen, DEVICE,
                                                model_type=attention_type)
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print(f"Prediction: {label_names[pred_label]} (confidence: {confidence:.2%})")

BERT Model Loaded
✓ Loaded ATTENTION model from: bert_absa_attention_model.pth

ATTENTION MODEL PREDICTIONS

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
Prediction: Positive (confidence: 100.00%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
Prediction: Negative (confidence: 99.90%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
Prediction: Positive (confidence: 98.50%)

Sentence: "I love the fast performance of this machine."
Aspect: "performance"
Prediction: Positive (confidence: 100.00%)

Sentence: "The customer service was unhelpful and rude."
Aspect: "customer service"
Prediction: Negative (confidence: 99.77%)


### 8.3 Base Model Inference

In [22]:
# Load Base model from saved file
base_model_path = 'bert_absa_base_model.pth'
base_model, base_maxlen, base_type = load_model(base_model_path, DEVICE)
print(f"✓ Loaded BASE model from: {base_model_path}")

print("\n" + "=" * 70)
print("BASE MODEL PREDICTIONS")
print("=" * 70)

for sentence, aspect in test_examples:
    pred_label, confidence = predict_sentiment(base_model, bert_tokenizer,
                                                sentence, aspect, base_maxlen, DEVICE,
                                                model_type=base_type)
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print(f"Prediction: {label_names[pred_label]} (confidence: {confidence:.2%})")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT Model Loaded
✓ Loaded BASE model from: bert_absa_base_model.pth

BASE MODEL PREDICTIONS

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
Prediction: Positive (confidence: 99.94%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
Prediction: Negative (confidence: 99.95%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
Prediction: Positive (confidence: 81.28%)

Sentence: "I love the fast performance of this machine."
Aspect: "performance"
Prediction: Positive (confidence: 99.94%)

Sentence: "The customer service was unhelpful and rude."
Aspect: "customer service"
Prediction: Negative (confidence: 99.94%)


### 8.4 Compare All Models

In [23]:
# Compare predictions from all 3 models side-by-side
print("=" * 80)
print("COMPARISON: ALL 3 MODELS ON SAME EXAMPLES")
print("=" * 80)

# Store loaded models for comparison
loaded_models = {
    'lstm': (lstm_model, lstm_maxlen, lstm_type),
    'attention': (attention_model, attention_maxlen, attention_type),
    'base': (base_model, base_maxlen, base_type)
}

for sentence, aspect in test_examples:
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print("-" * 70)
    
    for model_name, (model, max_len, model_type) in loaded_models.items():
        pred_label, confidence = predict_sentiment(model, bert_tokenizer,
                                                    sentence, aspect, max_len, DEVICE,
                                                    model_type=model_type)
        print(f"  {model_name.upper():<12}: {label_names[pred_label]:<10} (confidence: {confidence:.2%})")

COMPARISON: ALL 3 MODELS ON SAME EXAMPLES

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
----------------------------------------------------------------------
  LSTM        : Positive   (confidence: 99.50%)
  ATTENTION   : Positive   (confidence: 100.00%)
  BASE        : Positive   (confidence: 99.94%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
----------------------------------------------------------------------
  LSTM        : Negative   (confidence: 98.38%)
  ATTENTION   : Negative   (confidence: 99.90%)
  BASE        : Negative   (confidence: 99.95%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
----------------------------------------------------------------------
  LSTM        : Positive   (confidence: 99.34%)
  ATTENTION   : Positive   (confidence: 98.50%)
  BASE        : Positive   (confidence: 81.28%)

Sentence: "I love the fast performance of this m